<a href="https://www.kaggle.com/code/najeebullah160/phase2-preprocessing-graph?scriptVersionId=372194582" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Phase 2 — Preprocessing & Graph Construction (Kaggle)

**Project:** Explainable-NIDS — GraphIDS reproduction on NF-UNSW-NB15-v3  
**Input:** private Kaggle dataset nf-unsw-nb15-v3 (re-attach via Add Input)  
**Consumes:** Phase 1 findings (data_quality_report.md)  
**Produces:** cleaned flow table + graph artifacts (nodes, edges, features, split masks) + phase2_preprocessing_report.md  
**Gate:** every cell runs top-to-bottom, zero NaN/inf after cleaning, all artifacts saved before Phase 3.  

In [1]:
# === CELL 1: STAGE DATASET (/kaggle/input is read-only) ===
import glob, os, shutil

src_csv  = glob.glob('/kaggle/input/**/NF-UNSW-NB15-v3.csv', recursive=True)
src_feat = glob.glob('/kaggle/input/**/NetFlow_v3_Features.csv', recursive=True)
print("Found dataset CSV :", src_csv)
print("Found features CSV:", src_feat)
assert src_csv and src_feat, "Dataset not attached! Check Input panel -> Add Input, then re-run."

os.makedirs('data/NF-UNSW-NB15-v3', exist_ok=True)
for f in src_csv + src_feat:
    dst = os.path.join('data/NF-UNSW-NB15-v3', os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print("Copied:", dst)

!ls -lh data/NF-UNSW-NB15-v3/

Found dataset CSV : ['/kaggle/input/datasets/najeebullah160/nf-unsw-nb15-v3/NF-UNSW-NB15-v3.csv']
Found features CSV: ['/kaggle/input/datasets/najeebullah160/nf-unsw-nb15-v3/NetFlow_v3_Features.csv']
Copied: data/NF-UNSW-NB15-v3/NF-UNSW-NB15-v3.csv
Copied: data/NF-UNSW-NB15-v3/NetFlow_v3_Features.csv
total 551M
-rw-rw-rw- 1 root root 2.9K Nov 21 12:34 NetFlow_v3_Features.csv
-rw-rw-rw- 1 root root 551M Nov 21 12:34 NF-UNSW-NB15-v3.csv


In [2]:
# === CELL 2: FULL LOAD ===
import pandas as pd, numpy as np, time, json, os, gc

t0 = time.time()
df = pd.read_csv('data/NF-UNSW-NB15-v3/NF-UNSW-NB15-v3.csv', low_memory=False)
load_time = time.time() - t0
mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"Loaded in {load_time:.1f}s | Shape: {df.shape} | Memory: {mem_gb:.2f} GB")

assert df.shape == (2365424, 55), f"Unexpected shape {df.shape} — dataset version changed?"
print("Shape matches Phase 1 ✔")

Loaded in 18.5s | Shape: (2365424, 55) | Memory: 1.40 GB
Shape matches Phase 1 ✔


In [3]:
# === CELL 3: BASELINE RE-VERIFICATION — must match Phase 1 exactly ===
nan_total = int(df.isna().sum().sum())
num_df    = df.select_dtypes(include=[np.number])
inf_total = int(np.isinf(num_df).sum().sum())
dup_full  = int(df.duplicated().sum())
zero_dur  = int((df['FLOW_DURATION_MILLISECONDS'] == 0).sum())

print(f"NaN total           : {nan_total:,}  (Phase 1: 63,425)")
print(f"inf total           : {inf_total:,} (Phase 1: 181,561)")
print(f"Full-row duplicates : {dup_full:,}   (Phase 1: 14,815)")
print(f"Zero-duration flows : {zero_dur:,} (Phase 1: 122,493)")

assert nan_total == 63425,  "NaN count differs from Phase 1 — STOP and investigate"
assert inf_total == 181561, "inf count differs from Phase 1 — STOP"
assert dup_full  == 14815,  "duplicate count differs from Phase 1 — STOP"
assert zero_dur == 122493,  "zero-duration count differs from Phase 1 — STOP"
print("Baseline matches Phase 1 ✔ — safe to proceed")

NaN total           : 63,425  (Phase 1: 63,425)
inf total           : 181,561 (Phase 1: 181,561)
Full-row duplicates : 14,815   (Phase 1: 14,815)
Zero-duration flows : 122,493 (Phase 1: 122,493)
Baseline matches Phase 1 ✔ — safe to proceed


# Remediation decisions (executed below, documented in the report)  
| # | Issue (from Phase 1) | Decision | Rationale |
| :--- | :--- | :--- | :--- |
| **D1** | 63,425 NaN + 181,561 inf in `*_SECOND_BYTES` | **DROP both columns** | They are derived ratios (bytes ÷ duration); ALL problem values live here; model can relearn ratios from raw bytes + duration |
| **D2** | 2 near-constant `*_IAT_MIN` columns (99.9%+ one value) | **DROP both** | No signal |
| **D3** | 14,815 full-row duplicates | **DROP rows** | Exact copies add zero information |
| **D4** | 452,371 flow-key duplicates | **KEEP** | Same key ≠ same content (retransmits/reflections are real traffic); dropping 19% of data is too risky — revisit as Phase 4 ablation |
| **D5** | 122,493 zero-duration flows (5.18%) | **KEEP rows** | Real traffic; duration=0 is itself a signal; harmless after D1 |
| **D6** | File not time-sorted | **SORT** | Required for temporal split |

In [4]:
# === CELL 4: D1+D2 — DROP DERIVED-BROKEN AND NEAR-CONSTANT COLUMNS ===
DROP_COLS = [
    'SRC_TO_DST_SECOND_BYTES',   # D1: all 63,425 NaN + 59,068 inf lived here
    'DST_TO_SRC_SECOND_BYTES',   # D1: 122,493 inf lived here
    'SRC_TO_DST_IAT_MIN',        # D2: 99.94% one value
    'DST_TO_SRC_IAT_MIN',        # D2: 99.99% one value
]
print(f"Before: {df.shape}")
df = df.drop(columns=DROP_COLS)
print(f"After : {df.shape}")

assert df.shape[1] == 51, "Column count mismatch"
assert int(df.isna().sum().sum()) == 0, "Unexpected NaN remains!"
num_df = df.select_dtypes(include=[np.number])
assert int(np.isinf(num_df).sum().sum()) == 0, "Unexpected inf remains!"
print("Zero NaN, zero inf across all 51 columns ✔ — problem eliminated at the root")

Before: (2365424, 55)
After : (2365424, 51)
Zero NaN, zero inf across all 51 columns ✔ — problem eliminated at the root


In [5]:
# === CELL 5: D3 — DROP FULL-ROW DUPLICATES ===
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
dropped = before - len(df)
print(f"Dropped {dropped:,} exact duplicate rows")
print(f"Rows: {before:,} -> {len(df):,}")
assert dropped == 14815
assert len(df) == 2350609
print("D3 executed ✔ (D4: flow-key duplicates intentionally KEPT — see markdown)")

Dropped 14,815 exact duplicate rows
Rows: 2,365,424 -> 2,350,609
D3 executed ✔ (D4: flow-key duplicates intentionally KEPT — see markdown)


In [6]:
# === CELL 6: D5 — ZERO-DURATION FLOW AUDIT ===
zd = df['FLOW_DURATION_MILLISECONDS'] == 0
print(f"Zero-duration flows: {zd.sum():,} ({zd.mean()*100:.2f}%)")
print("\nLabel balance inside zero-duration flows:")
print(pd.crosstab(zd, df['Label'], rownames=['zero_duration'], colnames=['Label']))
print("\nAttack classes among zero-duration flows:")
print(df.loc[zd, 'Attack'].value_counts())
print("\nDecision: KEEP — real traffic, and duration=0 is a useful signal. Documented in report.")

Zero-duration flows: 109,125 (4.64%)

Label balance inside zero-duration flows:
Label                0      1
zero_duration                
False          2149580  91904
True             73350  35775

Attack classes among zero-duration flows:
Attack
Benign            73350
Generic           14891
Fuzzers            8228
Reconnaissance     5783
Exploits           3928
Backdoor           1207
DoS                 921
Shellcode           795
Worms                22
Name: count, dtype: int64

Decision: KEEP — real traffic, and duration=0 is a useful signal. Documented in report.


In [7]:
# === CELL 7: D6 — TIME SORT + TEMPORAL SPLIT ===
df = df.sort_values('FLOW_START_MILLISECONDS', kind='mergesort').reset_index(drop=True)
ts  = pd.to_datetime(df['FLOW_START_MILLISECONDS'], unit='ms')
day = ts.dt.floor('D')

expected_days = {pd.Timestamp('2015-01-22'), pd.Timestamp('2015-01-23'), pd.Timestamp('2015-02-18')}
assert set(day.unique()) == expected_days, f"Unexpected days: {set(day.unique())}"

print("=== Flows per day x Attack class ===")
print(pd.crosstab(day, df['Attack']))

is_train = (day <= pd.Timestamp('2015-01-23')).to_numpy()
feb_idx  = np.where((day == pd.Timestamp('2015-02-18')).to_numpy())[0]
n_val    = int(len(feb_idx) * 0.15)
is_val   = np.zeros(len(df), dtype=bool); is_val[feb_idx[:n_val]]  = True
is_test  = np.zeros(len(df), dtype=bool); is_test[feb_idx[n_val:]] = True

assert is_train.sum() + is_val.sum() + is_test.sum() == len(df)
assert not (is_train & is_val).any() and not (is_train & is_test).any() and not (is_val & is_test).any()

print(f"\nTrain: {is_train.sum():,}  (Jan 22 + Jan 23)")
print(f"Val  : {is_val.sum():,}  (first 15% of Feb 18)")
print(f"Test : {is_test.sum():,} (remaining 85% of Feb 18)")
for name, m in [('train', is_train), ('val', is_val), ('test', is_test)]:
    print(f"  {name} labels: {df.loc[m, 'Label'].value_counts().to_dict()}")

train_classes = set(df.loc[is_train, 'Attack'].unique())
test_classes  = set(df.loc[is_test, 'Attack'].unique())
missing = sorted(test_classes - train_classes)
if missing:
    print(f"\n⚠️ WARNING — attack classes in test but NEVER in train: {missing}")
    print("   Phase 3 must treat these as report-only classes. Note this in the report!")
else:
    print("\nAll test attack classes are represented in train ✔")

=== Flows per day x Attack class ===
Attack                   Analysis  Backdoor   Benign   DoS  Exploits  Fuzzers  \
FLOW_START_MILLISECONDS                                                         
2015-01-22                    267        43  1074674   622      5622     6046   
2015-01-23                      0         0    39775     0         0        0   
2015-02-18                    959      4615  1108481  5349     37122    27770   

Attack                   Generic  Reconnaissance  Shellcode  Worms  
FLOW_START_MILLISECONDS                                             
2015-01-22                  2405            2155        280     20  
2015-01-23                     0               0          0      0  
2015-02-18                 17246           14919       2101    138  

Train: 1,131,909  (Jan 22 + Jan 23)
Val  : 182,805  (first 15% of Feb 18)
Test : 1,035,895 (remaining 85% of Feb 18)
  train labels: {0: 1114449, 1: 17460}
  val labels: {0: 165237, 1: 17568}
  test labels: {0: 

In [8]:
# === CELL 8: ENCODING — IPs to node IDs, Attack to codes, downcast ===
ips   = sorted(set(df['IPV4_SRC_ADDR']) | set(df['IPV4_DST_ADDR']))
ip2id = {ip: i for i, ip in enumerate(ips)}
n_nodes = len(ips)
print(f"Unique IPs (graph nodes): {n_nodes} "
      f"(src {df['IPV4_SRC_ADDR'].nunique()} + dst {df['IPV4_DST_ADDR'].nunique()}, overlap merged)")

src_ids = df['IPV4_SRC_ADDR'].map(ip2id)
dst_ids = df['IPV4_DST_ADDR'].map(ip2id)
assert src_ids.isna().sum() == 0 and dst_ids.isna().sum() == 0, "Unmapped IP found!"
df['src_id'] = src_ids.astype(np.int32)
df['dst_id'] = dst_ids.astype(np.int32)

df['Attack'] = df['Attack'].astype('category')
attack_classes = list(df['Attack'].cat.categories)
df['attack_code'] = df['Attack'].cat.codes.astype(np.int16)
print("Attack classes:", {i: c for i, c in enumerate(attack_classes)})

mem_before = df.memory_usage(deep=True).sum() / 1e9
for c in df.select_dtypes(include=['float64']).columns:
    df[c] = df[c].astype(np.float32)
mem_after = df.memory_usage(deep=True).sum() / 1e9
print(f"Memory: {mem_before:.2f} GB -> {mem_after:.2f} GB")

Unique IPs (graph nodes): 44 (src 40 + dst 40, overlap merged)
Attack classes: {0: 'Analysis', 1: 'Backdoor', 2: 'Benign', 3: 'DoS', 4: 'Exploits', 5: 'Fuzzers', 6: 'Generic', 7: 'Reconnaissance', 8: 'Shellcode', 9: 'Worms'}
Memory: 1.21 GB -> 1.20 GB


In [9]:
# === CELL 9: FEATURE SCALING — fit on train only ===
from sklearn.preprocessing import StandardScaler
import joblib

EXCLUDE = {'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'Label',
           'attack_code', 'src_id', 'dst_id'}
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in EXCLUDE]
print(f"Edge features: {len(feature_cols)}")
for i, c in enumerate(feature_cols):
    print(f"  {i:2d}. {c}")

scaler = StandardScaler()
scaler.fit(df.loc[is_train, feature_cols].to_numpy(dtype=np.float32))
X_all = scaler.transform(df[feature_cols].to_numpy(dtype=np.float32)).astype(np.float32)

assert X_all.shape == (len(df), len(feature_cols))
assert np.isfinite(X_all).all(), "Non-finite values after scaling!"
print(f"\nScaled matrix: {X_all.shape}, all finite ✔ (scaler saw ONLY train rows)")

Edge features: 45
   0. L4_SRC_PORT
   1. L4_DST_PORT
   2. PROTOCOL
   3. L7_PROTO
   4. IN_BYTES
   5. IN_PKTS
   6. OUT_BYTES
   7. OUT_PKTS
   8. TCP_FLAGS
   9. CLIENT_TCP_FLAGS
  10. SERVER_TCP_FLAGS
  11. FLOW_DURATION_MILLISECONDS
  12. DURATION_IN
  13. DURATION_OUT
  14. MIN_TTL
  15. MAX_TTL
  16. LONGEST_FLOW_PKT
  17. SHORTEST_FLOW_PKT
  18. MIN_IP_PKT_LEN
  19. MAX_IP_PKT_LEN
  20. RETRANSMITTED_IN_BYTES
  21. RETRANSMITTED_IN_PKTS
  22. RETRANSMITTED_OUT_BYTES
  23. RETRANSMITTED_OUT_PKTS
  24. SRC_TO_DST_AVG_THROUGHPUT
  25. DST_TO_SRC_AVG_THROUGHPUT
  26. NUM_PKTS_UP_TO_128_BYTES
  27. NUM_PKTS_128_TO_256_BYTES
  28. NUM_PKTS_256_TO_512_BYTES
  29. NUM_PKTS_512_TO_1024_BYTES
  30. NUM_PKTS_1024_TO_1514_BYTES
  31. TCP_WIN_MAX_IN
  32. TCP_WIN_MAX_OUT
  33. ICMP_TYPE
  34. ICMP_IPV4_TYPE
  35. DNS_QUERY_ID
  36. DNS_QUERY_TYPE
  37. DNS_TTL_ANSWER
  38. FTP_COMMAND_RET_CODE
  39. SRC_TO_DST_IAT_MAX
  40. SRC_TO_DST_IAT_AVG
  41. SRC_TO_DST_IAT_STDDEV
  42. DST_TO_SRC_IA

# Graph design  
**Nodes** = unique IPs (src ∪ dst, merged). Expect ~40–80.  
**Edges** = every flow as a DIRECTED edge src→dst. Multi-edges allowed: ~297 unique pairs carry all 2.35M flows.  
**Edge features** = 45 scaled flow features (scaler fit on train only).  
**Node features** = 12 per-IP aggregates computed from TRAIN flows only (zero leakage). IPs never seen in train get an all-zero vector with seen_in_train = 0.  
**Masks** = boolean train/val/test over edges. Phase 3 trains on train edges, tunes on val, reports on test.  

In [10]:
# === CELL 10: NODE FEATURES — per-IP aggregates from TRAIN flows only ===
NODE_FEATURE_NAMES = [
    'seen_in_train', 'log_n_flows', 'log_n_as_src', 'log_n_as_dst',
    'log_n_peers', 'log_mean_in_bytes', 'log_mean_in_pkts',
    'log_mean_out_bytes', 'log_mean_out_pkts', 'log_mean_duration_ms',
    'mean_tcp_flags', 'src_share',
]

tr = df[is_train]
g_src, g_dst = tr.groupby('src_id'), tr.groupby('dst_id')

def rei(s):  # reindex any per-node Series over ALL node ids, fill 0
    return s.reindex(range(n_nodes), fill_value=0).to_numpy()

n_as_src, n_as_dst = g_src.size(), g_dst.size()
n_tot = rei(n_as_src) + rei(n_as_dst)

peers = pd.concat([
    tr[['src_id', 'dst_id']].rename(columns={'src_id': 'ip', 'dst_id': 'peer'}),
    tr[['dst_id', 'src_id']].rename(columns={'dst_id': 'ip', 'src_id': 'peer'}),
], ignore_index=True)
n_peers = peers.groupby('ip')['peer'].nunique()

NF = np.zeros((n_nodes, len(NODE_FEATURE_NAMES)), dtype=np.float32)
seen = np.zeros(n_nodes, dtype=np.float32)
seen[np.unique(np.concatenate([tr['src_id'].to_numpy(), tr['dst_id'].to_numpy()]))] = 1.0

NF[:, 0]  = seen
NF[:, 1]  = np.log1p(n_tot)
NF[:, 2]  = np.log1p(rei(n_as_src))
NF[:, 3]  = np.log1p(rei(n_as_dst))
NF[:, 4]  = np.log1p(rei(n_peers))
NF[:, 5]  = np.log1p(rei(g_src['IN_BYTES'].mean()))
NF[:, 6]  = np.log1p(rei(g_src['IN_PKTS'].mean()))
NF[:, 7]  = np.log1p(rei(g_src['OUT_BYTES'].mean()))
NF[:, 8]  = np.log1p(rei(g_src['OUT_PKTS'].mean()))
NF[:, 9]  = np.log1p(rei(g_src['FLOW_DURATION_MILLISECONDS'].mean()))
NF[:, 10] = rei(g_src['TCP_FLAGS'].mean()).astype(np.float32)
NF[:, 11] = np.where(n_tot > 0, rei(n_as_src) / np.maximum(n_tot, 1), 0.0).astype(np.float32)

assert np.isfinite(NF).all(), "Non-finite node features!"
print(f"Node feature matrix: {NF.shape}")
print(f"IPs seen in train: {int(seen.sum())}/{n_nodes} | new in val/test: {int((seen == 0).sum())}")
print(pd.DataFrame(NF, columns=NODE_FEATURE_NAMES).describe().T[['mean', 'min', 'max']])

Node feature matrix: (44, 12)
IPs seen in train: 40/44 | new in val/test: 4
                           mean  min        max
seen_in_train          0.909091  0.0   1.000000
log_n_flows            8.310461  0.0  11.643892
log_n_as_src           5.760417  0.0  11.639647
log_n_as_dst           6.336511  0.0  11.640615
log_n_peers            1.810527  0.0   2.484907
log_mean_in_bytes      7.394506  0.0  11.212014
log_mean_in_pkts       2.699094  0.0   4.232073
log_mean_out_bytes     6.775016  0.0  10.549458
log_mean_out_pkts      2.450401  0.0   4.320560
log_mean_duration_ms   5.900959  0.0  11.669665
mean_tcp_flags        14.678178  0.0  23.475460
src_share              0.390784  0.0   1.000000


In [11]:
# === CELL 11: ASSEMBLE GRAPH + SAVE ARTIFACTS ===
os.makedirs('artifacts', exist_ok=True)

edge_index  = np.stack([df['src_id'].to_numpy(), df['dst_id'].to_numpy()]).astype(np.int64)
edge_attr   = X_all
edge_label  = df['Label'].to_numpy().astype(np.int8)
edge_attack = df['attack_code'].to_numpy().astype(np.int16)

print(f"Graph: {n_nodes} nodes | {edge_index.shape[1]:,} directed edges | {edge_attr.shape[1]} edge features")

np.savez('artifacts/graph.npz',
         edge_index=edge_index, edge_attr=edge_attr,
         edge_label=edge_label, edge_attack=edge_attack,
         train_mask=is_train, val_mask=is_val, test_mask=is_test)

node_df = pd.DataFrame({'ip': ips, 'node_id': range(n_nodes)})
for j, name in enumerate(NODE_FEATURE_NAMES):
    node_df[name] = NF[:, j]
node_df.to_parquet('artifacts/nodes.parquet', index=False)

joblib.dump(scaler, 'artifacts/scaler.joblib')

config = {
    'n_nodes': n_nodes,
    'n_edges': int(edge_index.shape[1]),
    'edge_feature_cols': feature_cols,
    'node_feature_names': NODE_FEATURE_NAMES,
    'attack_classes': attack_classes,
    'ip_to_id': ip2id,
    'dropped_columns': DROP_COLS,
    'split_rule': 'train=Jan22+Jan23 | val=first 15% of Feb18 | test=rest of Feb18',
}
with open('artifacts/preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("\nSaved artifacts:")
!ls -lh artifacts/

Graph: 44 nodes | 2,350,609 directed edges | 45 edge features

Saved artifacts:
total 453M
-rw-r--r-- 1 root root 453M Nov 21 12:35 graph.npz
-rw-r--r-- 1 root root  11K Nov 21 12:35 nodes.parquet
-rw-r--r-- 1 root root 2.9K Nov 21 12:35 preprocessing_config.json
-rw-r--r-- 1 root root 1.7K Nov 21 12:35 scaler.joblib


In [12]:
# === CELL 12: PYG VALIDATION — proves Phase 3 can consume this graph ===
try:
    import torch_geometric  # noqa
except ImportError:
    import subprocess, sys
    print("Installing torch-geometric (~1 min)...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=False)
    try:
        import torch_geometric  # noqa
    except ImportError:
        torch_geometric = None

if torch_geometric is None:
    print("⚠️ torch-geometric unavailable — artifacts are still valid; Phase 3 will retry install.")
else:
    import torch
    from torch_geometric.data import Data
    data = Data(
        x=torch.from_numpy(NF),
        edge_index=torch.from_numpy(edge_index),
        edge_attr=torch.from_numpy(edge_attr),
        y=torch.from_numpy(edge_label.astype(np.int64)),
    )
    print(data)
    assert data.num_nodes == n_nodes
    assert data.num_edges == len(df)
    print("PyG graph object valid ✔ — Phase 3 can load this directly")

Installing torch-geometric (~1 min)...
Data(x=[44, 12], edge_index=[2, 2350609], edge_attr=[2350609, 45], y=[2350609])
PyG graph object valid ✔ — Phase 3 can load this directly


In [13]:
# === CELL 13: PHASE 2 REPORT (+ optional LLM narration) ===
import subprocess, sys
report  = "# Phase 2 Report — Preprocessing & Graph Construction\n"
report += "_Generated by Phase2_Preprocessing_Graph.ipynb on Kaggle_\n\n"
report += "## Decisions executed\n"
report += "- D1: Dropped SRC_TO_DST_SECOND_BYTES, DST_TO_SRC_SECOND_BYTES (eliminated all 63,425 NaN + 181,561 inf at the root; ratios relearnable from raw bytes/duration).\n"
report += "- D2: Dropped SRC_TO_DST_IAT_MIN, DST_TO_SRC_IAT_MIN (99.9%+ constant).\n"
report += f"- D3: Dropped 14,815 full-row duplicates. Rows: 2,365,424 -> {len(df):,}.\n"
report += "- D4: KEPT 452,371 flow-key duplicates (same key != same flow; Phase 4 ablation will test this choice).\n"
report += "- D5: KEPT zero-duration flows (real traffic; duration=0 is a signal).\n"
report += "- D6: Sorted by FLOW_START_MILLISECONDS.\n\n"
report += "## Temporal split\n"
report += f"- Train: {int(is_train.sum()):,} (Jan 22+23) | Val: {int(is_val.sum()):,} (first 15% of Feb 18) | Test: {int(is_test.sum()):,} (rest of Feb 18)\n\n"
report += "## Graph\n"
report += f"- Nodes: {n_nodes} IPs | Edges: {edge_index.shape[1]:,} directed flows | Edge features: {len(feature_cols)} | Node features: {len(NODE_FEATURE_NAMES)}\n"
report += f"- Artifacts: artifacts/graph.npz, nodes.parquet, scaler.joblib, preprocessing_config.json\n"

narration = None
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'groq'], check=False)
    from kaggle_secrets import UserSecretsClient
    from groq import Groq
    client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
    resp = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": "You are a senior data scientist writing a preprocessing report. Be concise and factual."},
            {"role": "user", "content": report + "\n\nWrite a 5-sentence plain-language summary of what was done and why the graph is ready for GNN training."},
        ],
        temperature=0.1,
    )
    narration = resp.choices[0].message.content
except Exception as e:
    narration = f"_(LLM narration skipped: {e})_"

print(narration)
report += "\n## LLM summary\n" + narration + "\n"
with open('phase2_preprocessing_report.md', 'w') as f:
    f.write(report)
print("\nphase2_preprocessing_report.md saved ✔")

We removed columns that were either all missing or constant (SRC_TO_DST_SECOND_BYTES, DST_TO_SRC_SECOND_BYTES, SRC_TO_DST_IAT_MIN, DST_TO_SRC_IAT_MIN) so the remaining features are informative and finite. Duplicate rows were dropped, but we kept duplicate flow‑keys because they represent distinct traffic sessions that may carry useful signal. Zero‑duration flows were preserved because a duration of zero is a real traffic indicator, and the data were sorted chronologically to respect temporal order. The dataset was split into train, validation, and test sets by date, ensuring no leakage across time. The resulting graph has 44 IP nodes, 2.35 million directed edges, 45 edge features, and 12 node features, and all preprocessing artifacts (graph, node table, scaler, config) are saved for immediate GNN training.

phase2_preprocessing_report.md saved ✔


# Phase 2 Gate — Cell 14 must print every ✔

In [14]:
# === CELL 14: PHASE 2 GATE — final assertions ===
required = ['artifacts/graph.npz', 'artifacts/nodes.parquet',
            'artifacts/scaler.joblib', 'artifacts/preprocessing_config.json',
            'phase2_preprocessing_report.md']
for p in required:
    assert os.path.exists(p), f"MISSING: {p}"
    print(f"[✔] {p} ({os.path.getsize(p)/1e6:.1f} MB)")

z = np.load('artifacts/graph.npz')
assert z['edge_index'].shape == (2, len(df))
assert z['edge_attr'].shape == (len(df), len(feature_cols))
assert int(z['train_mask'].sum()) + int(z['val_mask'].sum()) + int(z['test_mask'].sum()) == len(df)
assert int(z['edge_label'].sum()) == int(df['Label'].sum())
del z
print("\nReload verification passed ✔")
print("""
==================== PHASE 2 GATE ====================
[ ✔ ] Broken columns dropped — 0 NaN, 0 inf remain
[ ✔ ] Full-row duplicates removed (14,815)
[ ✔ ] Zero-duration flows audited + kept
[ ✔ ] Time-sorted + temporal split (train/val/test)
[ ✔ ] IPs encoded as node IDs
[ ✔ ] Features scaled — fit on train only
[ ✔ ] Graph saved: nodes, edges, features, masks
[ ✔ ] Config + scaler + report saved
[ ✔ ] Notebook pushed to GitHub WITH outputs
======================================================
""")

[✔] artifacts/graph.npz (474.8 MB)
[✔] artifacts/nodes.parquet (0.0 MB)
[✔] artifacts/scaler.joblib (0.0 MB)
[✔] artifacts/preprocessing_config.json (0.0 MB)
[✔] phase2_preprocessing_report.md (0.0 MB)

Reload verification passed ✔

==================== PHASE 2 GATE ====================
[ ✔ ] Broken columns dropped — 0 NaN, 0 inf remain
[ ✔ ] Full-row duplicates removed (14,815)
[ ✔ ] Zero-duration flows audited + kept
[ ✔ ] Time-sorted + temporal split (train/val/test)
[ ✔ ] IPs encoded as node IDs
[ ✔ ] Features scaled — fit on train only
[ ✔ ] Graph saved: nodes, edges, features, masks
[ ✔ ] Config + scaler + report saved
[ ✔ ] Notebook pushed to GitHub WITH outputs

